In [2]:
import pandas as pd 
import duckdb

In [3]:
import plotly.express as px

In [38]:
conn=duckdb.connect(r"D:\SEC.gov project\database\credit_risk.db")

In [39]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def dsql(line, cell):
    df = conn.sql(cell).df()
    var_name = line.strip()
    if var_name:
        globals()[var_name] = df
    return df

EDA 01 — Dataset Profile

Analytical Population

In [7]:
%%dsql profile_df
SELECT
    COUNT(DISTINCT s.cik) AS number_of_companies,
    COUNT(DISTINCT g.accession_number) AS total_filings,
    COUNT(*) AS total_observations,
    MIN(g.period_end) AS first_period,
    MAX(g.period_end) AS last_period
FROM gold.financial_metrics g
JOIN clean.sub s
    ON g.accession_number = s.adsh;

,number_of_companies,total_filings,total_observations,first_period,last_period
0,140,2573,10085,2014-12-31,2025-11-30


Total Observations By Year

In [9]:
%%dsql 
SELECT 
    EXTRACT(YEAR FROM period_end) AS year,
    COUNT(*) AS total_observations
FROM gold.financial_metrics
GROUP BY year
ORDER BY year;

,year,total_observations
0,2014,1
1,2015,1
2,2016,49
3,2017,297
4,2018,1136
5,2019,1467
6,2020,1402
7,2021,1422
8,2022,1416
9,2023,1339


Observation volume is highly uneven across the time series: 2019–2024 provide roughly 1.2k–1.5k observations per year, while 2025 has only 392. Therefore, we should not interpret 2025 as directly comparable to prior full years without accounting for incomplete-year coverage.

In [18]:
%%dsql
SELECT *
FROM clean.sub 
WHERE adsh IN (
    SELECT accession_number 
    FROM gold.financial_metrics 
    WHERE EXTRACT(YEAR FROM period_end) < 2019 
)
AND (EXTRACT(YEAR FROM period) < 2019)

,adsh,cik,name,sic,fye,form,period,fy,fp,filed,prevrpt
0,0001140361-19-000589,918251,MOTORCAR PARTS AMERICA INC,3714,331,10-Q,2018-09-30,2019,Q2,2019-01-08,0
1,0000775158-19-000006,775158,OSHKOSH CORP,3711,930,10-Q,2018-12-31,2019,Q1,2019-01-31,0
2,0001437749-19-001882,7623,ARTS WAY MANUFACTURING CO INC,3523,1130,10-K,2018-11-30,2018,FY,2019-02-05,0
3,0001193125-19-067169,95552,SUPERIOR INDUSTRIES INTERNATIONAL INC,3714,1231,10-K,2018-12-31,2018,FY,2019-03-07,0
4,0001477932-19-000995,1451512,TERRA TECH CORP.,3510,1231,10-K,2018-12-31,2018,FY,2019-03-15,0
...,...,...,...,...,...,...,...,...,...,...,...
79,0000277509-19-000008,277509,FEDERAL SIGNAL CORP /DE/,3711,1231,10-K,2018-12-31,2018,FY,2019-02-28,0
80,0001493152-19-010838,1586495,BLOW & DRIVE INTERLOCK CORP,3714,1231,10-K,2018-12-31,2018,FY,2019-07-19,0
81,0001213900-19-005483,714284,SORL AUTO PARTS INC,3714,1231,10-K,2018-12-31,2018,FY,2019-04-01,0
82,0001564590-19-006097,1455684,"TPI COMPOSITES, INC",3510,1231,10-K,2018-12-31,2018,FY,2019-03-05,0


In [19]:
%%dsql
SELECT *
FROM clean.sub 
WHERE adsh IN (
    SELECT accession_number 
    FROM gold.financial_metrics 
    WHERE EXTRACT(YEAR FROM period_end) < 2019 
)
AND (EXTRACT(YEAR FROM filed) < 2019)

,adsh,cik,name,sic,fye,form,period,fy,fp,filed,prevrpt


In [20]:
%%dsql
SELECT
    MIN(filed) AS first_filed,
    MAX(filed) AS last_filed
FROM clean.sub;

,first_filed,last_filed
0,2019-01-08,2025-12-19


Temporal scope validation: The Gold table contains 84 observations with period_end < 2019, but their corresponding filings were all submitted in 2019 or later. clean.sub has a filing-date range of 2019-01-08 to 2025-12-19. Therefore, the project's 2019–2025 population is correctly defined by filing date, and pre-2019 period_end observations should not be removed from Gold. filed should be retained alongside period_end to preserve both disclosure timing and financial-period context.

In [36]:
conn.sql("""
UPDATE gold.financial_metrics AS g
SET filed = s.filed
FROM clean.sub AS s
WHERE g.accession_number = s.adsh;
""")

Observations By SIC

In [23]:
%%dsql
SELECT 
    s.sic,
    COUNT(DISTINCT s.cik) AS total_companies,
    COUNT(DISTINCT g.accession_number) AS total_filings,
    ROUND(COUNT(*)*100/(SELECT COUNT(*) FROM gold.financial_metrics), 2) AS observation_percentage
FROM gold.financial_metrics g
JOIN clean.sub s
ON g.accession_number = s.adsh
GROUP BY s.sic
ORDER BY observation_percentage DESC

,sic,total_companies,total_filings,observation_percentage
0,3714,63,1194,47.09
1,3711,36,525,19.46
2,3510,12,234,9.80
3,3531,10,188,7.19
4,3523,10,187,6.33
5,3713,6,105,4.21
6,3537,2,56,2.26
7,3715,1,28,1.87
8,3716,2,56,1.78


While doing peer/industry analysis, we should compare metrics within SIC, rather than treating all 10,085 observations as an equally representative cross-industry population.

Observations By Filing Type

In [11]:
%%dsql 
SELECT 
    s.form,
    COUNT(*) AS total_observations
FROM gold.financial_metrics g 
JOIN clean.sub s 
ON g.accession_number = s.adsh
GROUP BY s.form
ORDER BY total_observations DESC;

,form,total_observations
0,10-Q,7519
1,10-K,2566


Metric Coverage

In [22]:
%%dsql
SELECT 
    column_name AS metric,
    100-null_percentage as coverage
FROM (SUMMARIZE TABLE gold.financial_metrics)t
ORDER BY coverage ASC;


,metric,coverage
0,fcf_to_debt_ratio,3.51
1,ocf_debt_ratio,3.80
2,quick_ratio,10.08
3,return_on_assets,24.83
4,interest_coverage_ratio,33.11
5,short_term_debt_ratio,36.39
6,current_ratio,49.19
7,net_working_capital,49.19
8,cash_ratio,49.41
9,debt_to_assets_ratio,54.37


Metric availability is highly heterogeneous. Therefore, subsequent analysis must be metric-specific rather than assuming that all Gold metrics describe the same analytical population. Coverage will be considered when deciding whether a metric can support population-wide analysis, subgroup comparisons, trends, or composite scoring.

Summary Statistics of Metrics

In [14]:
%%dsql
SELECT 
    column_name AS metric,
    ROUND(CAST(min AS NUMERIC),2) AS min,
    ROUND(CAST(max AS NUMERIC),2) AS max,
    ROUND(CAST(avg AS NUMERIC),2) AS mean,
    ROUND(CAST(std AS NUMERIC),2) AS std_dev,
    ROUND(CAST(q25 AS NUMERIC),2) AS q25,
    ROUND(CAST(q50 AS NUMERIC),2) AS median,
    ROUND(CAST(q75 AS FLOAT),2) AS q75
FROM (SUMMARIZE TABLE gold.financial_metrics)t
WHERE column_name NOT IN ('accession_number','period_end','filed')
ORDER BY min DESC;

,metric,min,max,mean,std_dev,q25,median,q75
0,current_ratio,0.00,"1,959.00",7.83,86.81,1.27,1.72,2.49
1,cash_ratio,0.00,"1,959.00",5.75,81.55,0.08,0.27,0.61
2,debt_to_assets_ratio,0.00,7.65,0.14,0.25,0.00,0.01,0.23
3,short_term_debt_ratio,0.00,1.00,0.23,0.34,0.01,0.04,0.27
4,quick_ratio,-0.19,82.47,2.81,5.63,0.85,1.27,1.83
5,fcf_to_debt_ratio,-50.46,11.55,-0.74,4.21,-0.13,-0.03,0.02
6,return_on_assets,-219.10,73.20,-0.36,6.16,-0.03,0.01,0.02
7,cash_earnings_conversion,"-1,644.50",722.39,0.04,28.79,0.00,0.00,0.00
8,ocf_debt_ratio,"-2,272.00",39.83,-24.69,213.40,-0.20,-0.01,0.06
9,interest_coverage_ratio,"-59,758.60","10,319.96",-70.63,"1,521.16",-1.59,1.61,8.46


Distribution & aggregation decision: Financial ratios exhibit substantial skewness and extreme tails, with means frequently diverging materially from medians. Consequently, median and percentile-based summaries will be preferred for descriptive, peer-comparison and trend analyses; means will be used only where analytically justified. Extreme observations will not be removed solely on the basis of magnitude.

In [40]:
pd.set_option('display.float_format', '{:,.2f}'.format)

In [27]:
df=conn.sql("""
    WITH quantile_thresholds AS (
        SELECT 
            PERCENTILE_CONT(0.01) WITHIN GROUP (ORDER BY current_ratio) AS lower_quantile,
            PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY current_ratio) AS upper_quantile
        FROM gold.financial_metrics
        WHERE current_ratio IS NOT NULL
        
    )
    SELECT 
        accession_number,
        period_end,
        filed,
        current_ratio 
    FROM gold.financial_metrics, quantile_thresholds
    WHERE current_ratio > lower_quantile
    AND current_ratio < upper_quantile
""")
fig=px.histogram(
    df,
    x='current_ratio',
    nbins=100,
    title='Distribution of Current Ratio',
    labels={'current_ratio':'Current Ratio'},
    hover_data={'accession_number':True, 'period_end':True, 'filed':True},
    template='plotly_dark'
)
fig.show()

In [28]:
df=conn.sql("""
    WITH quantile_thresholds AS (
        SELECT 
            PERCENTILE_CONT(0.01) WITHIN GROUP (ORDER BY cash_ratio) AS lower_quantile,
            PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY cash_ratio) AS upper_quantile
        FROM gold.financial_metrics
        WHERE cash_ratio IS NOT NULL
        
    )
    SELECT 
        accession_number,
        period_end,
        filed,
        cash_ratio 
    FROM gold.financial_metrics, quantile_thresholds
    WHERE cash_ratio > lower_quantile
    AND cash_ratio < upper_quantile
""")
fig=px.histogram(
    df,
    x='cash_ratio',
    nbins=100,
    title='Distribution of Cash Ratio',
    labels={'cash_ratio':'Cash Ratio'},
    hover_data={'accession_number':True, 'period_end':True, 'filed':True},
    template='plotly_dark'
)
fig.show()

In [29]:
df=conn.sql("""
    WITH quantile_thresholds AS (
        SELECT 
            PERCENTILE_CONT(0.01) WITHIN GROUP (ORDER BY debt_to_assets_ratio) AS lower_quantile,
            PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY debt_to_assets_ratio) AS upper_quantile
        FROM gold.financial_metrics
        WHERE debt_to_assets_ratio IS NOT NULL
        
    )
    SELECT 
        accession_number,
        period_end,
        filed,
        debt_to_assets_ratio 
    FROM gold.financial_metrics, quantile_thresholds
    WHERE debt_to_assets_ratio > lower_quantile
    AND debt_to_assets_ratio < upper_quantile
""")
fig=px.histogram(
    df,
    x='debt_to_assets_ratio',
    nbins=100,
    title='Distribution of Debt to Assets Ratio',
    labels={'debt_to_assets_ratio':'Debt to Assets Ratio'},
    hover_data={'accession_number':True, 'period_end':True, 'filed':True},
    template='plotly_dark'
)
fig.show()

In [24]:
df=conn.sql("""
    WITH quantile_thresholds AS (
        SELECT 
            PERCENTILE_CONT(0.01) WITHIN GROUP (ORDER BY short_term_debt_ratio) AS lower_quantile,
            PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY short_term_debt_ratio) AS upper_quantile
        FROM gold.financial_metrics
        WHERE short_term_debt_ratio IS NOT NULL
        
    )
    SELECT 
        accession_number,
        period_end,
        filed,
        short_term_debt_ratio 
    FROM gold.financial_metrics, quantile_thresholds
    WHERE short_term_debt_ratio > lower_quantile
    AND short_term_debt_ratio < upper_quantile
""")
fig=px.histogram(
    df,
    x='short_term_debt_ratio',
    nbins=100,
    title='Distribution of Short-term Debt Ratio',
    labels={'short_term_debt_ratio':'Short-term Debt Ratio'},
    hover_data={'accession_number':True, 'period_end':True, 'filed':True},
    template='plotly_dark'
)
fig.show()

In [23]:
df=conn.sql("""
    WITH quantile_thresholds AS (
        SELECT 
            PERCENTILE_CONT(0.01) WITHIN GROUP (ORDER BY quick_ratio) AS lower_quantile,
            PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY quick_ratio) AS upper_quantile
        FROM gold.financial_metrics
        WHERE quick_ratio IS NOT NULL
        
    )
    SELECT 
        accession_number,
        period_end,
        filed,
        quick_ratio 
    FROM gold.financial_metrics, quantile_thresholds
    WHERE quick_ratio > lower_quantile
    AND quick_ratio < upper_quantile
""")
fig=px.histogram(
    df,
    x='quick_ratio',
    nbins=100,
    title='Distribution of Quick Ratio',
    labels={'quick_ratio':'Quick Ratio'},
    hover_data={'accession_number':True, 'period_end':True, 'filed':True},
    template='plotly_dark',
    width=1450
)
fig.show()

In [51]:
df=conn.sql("""
    WITH quantile_thresholds AS (
        SELECT 
            PERCENTILE_CONT(0.01) WITHIN GROUP (ORDER BY fcf_to_debt_ratio) AS lower_quantile,
            PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY fcf_to_debt_ratio) AS upper_quantile
        FROM gold.financial_metrics
        WHERE fcf_to_debt_ratio IS NOT NULL
        
    )
    SELECT 
        accession_number,
        period_end,
        filed,
        fcf_to_debt_ratio 
    FROM gold.financial_metrics, quantile_thresholds
    WHERE fcf_to_debt_ratio > lower_quantile
    AND fcf_to_debt_ratio < upper_quantile
""")
fig=px.histogram(
    df,
    x='fcf_to_debt_ratio',
    nbins=100,
    title='Distribution of FCF to Debt Ratio',
    labels={'fcf_to_debt_ratio':'FCF to Debt Ratio'},
    hover_data={'accession_number':True, 'period_end':True, 'filed':True},
    template='plotly_dark'
)
fig.show()

In [36]:
df=conn.sql("""
    WITH quantile_thresholds AS (
        SELECT 
            PERCENTILE_CONT(0.01) WITHIN GROUP (ORDER BY return_on_assets) AS lower_quantile,
            PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY return_on_assets) AS upper_quantile
        FROM gold.financial_metrics
        WHERE return_on_assets	IS NOT NULL
        
    )
    SELECT 
        accession_number,
        period_end,
        filed,
        return_on_assets 
    FROM gold.financial_metrics, quantile_thresholds
    WHERE return_on_assets > lower_quantile
    AND return_on_assets < upper_quantile
""")
fig=px.histogram(
    df,
    x='return_on_assets',
    nbins=100,
    title='Distribution of Return on Assets',
    labels={'return_on_assets':'Return on Assets'},
    hover_data={'accession_number':True, 'period_end':True, 'filed':True},
    template='plotly_dark'
)
fig.show()

In [38]:
df=conn.sql("""
    WITH quantile_thresholds AS (
        SELECT 
            PERCENTILE_CONT(0.01) WITHIN GROUP (ORDER BY cash_earnings_conversion) AS lower_quantile,
            PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY cash_earnings_conversion) AS upper_quantile
        FROM gold.financial_metrics
        WHERE cash_earnings_conversion IS NOT NULL
        
    )
    SELECT 
        accession_number,
        period_end,
        filed,
        cash_earnings_conversion 
    FROM gold.financial_metrics, quantile_thresholds
    WHERE cash_earnings_conversion > lower_quantile
    AND cash_earnings_conversion < upper_quantile
""")
fig=px.histogram(
    df,
    x='cash_earnings_conversion',
    nbins=100,
    title='Distribution of Cash Earnings Conversion',
    labels={'cash_earnings_conversion':'Cash Earnings Conversion'},
    hover_data={'accession_number':True, 'period_end':True, 'filed':True},
    template='plotly_dark'
)
fig.show()

In [33]:
df=conn.sql("""
    WITH quantile_thresholds AS (
        SELECT 
            PERCENTILE_CONT(0.01) WITHIN GROUP (ORDER BY ocf_debt_ratio) AS lower_quantile,
            PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY ocf_debt_ratio) AS upper_quantile
        FROM gold.financial_metrics
        WHERE ocf_debt_ratio IS NOT NULL
        
    )
    SELECT 
        accession_number,
        period_end,
        filed,
        ocf_debt_ratio 
    FROM gold.financial_metrics, quantile_thresholds
    WHERE ocf_debt_ratio > lower_quantile
    AND ocf_debt_ratio < upper_quantile
""")
fig=px.histogram(
    df,
    x='ocf_debt_ratio',
    nbins=100,
    title='Distribution of OCF/Debt Ratio',
    labels={'ocf_debt_ratio':'OCF/Debt Ratio'},
    hover_data={'accession_number':True, 'period_end':True, 'filed':True},
    template='plotly_dark'
)
fig.show()

In [34]:
df=conn.sql("""
    WITH quantile_thresholds AS (
        SELECT 
            PERCENTILE_CONT(0.01) WITHIN GROUP (ORDER BY interest_coverage_ratio) AS lower_quantile,
            PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY interest_coverage_ratio) AS upper_quantile
        FROM gold.financial_metrics
        WHERE interest_coverage_ratio IS NOT NULL
        
    )
    SELECT 
        accession_number,
        period_end,
        filed,
        interest_coverage_ratio 
    FROM gold.financial_metrics, quantile_thresholds
    WHERE interest_coverage_ratio > lower_quantile
    AND interest_coverage_ratio < upper_quantile
""")
fig=px.histogram(
    df,
    x='interest_coverage_ratio',
    nbins=100,
    title='Distribution of Interest Coverage Ratio',
    labels={'interest_coverage_ratio':'Interest Coverage Ratio'},
    hover_data={'accession_number':True, 'period_end':True, 'filed':True},
    template='plotly_dark'
)
fig.show()

Looking into the extreme values of some metrics.

Current Ratio

In [70]:
%%dsql 
SELECT DISTINCT 
    s.cik,
    s.name,
    accession_number,
    period_end,
    g.filed,
    n.tag,
    n.value,
    n.uom
FROM gold.financial_metrics g 
JOIN clean.num n 
ON g.accession_number = n.adsh
AND g.period_end=n.ddate
JOIN clean.sub s
ON g.accession_number = s.adsh
WHERE current_ratio = (SELECT MAX(current_ratio) FROM gold.financial_metrics) 
AND n.tag IN ('AssetsCurrent', 'LiabilitiesCurrent')

,cik,name,accession_number,period_end,filed,tag,value,uom
0,1703157,"SECURETECH INNOVATIONS, INC.",0001703157-19-000012,2018-12-31,2019-05-16,AssetsCurrent,"195,900.00",USD
1,1703157,"SECURETECH INNOVATIONS, INC.",0001703157-19-000003,2018-12-31,2019-02-19,LiabilitiesCurrent,100.00,USD
2,1703157,"SECURETECH INNOVATIONS, INC.",0001703157-19-000017,2018-12-31,2019-10-21,LiabilitiesCurrent,100.00,USD
3,1703157,"SECURETECH INNOVATIONS, INC.",0001703157-19-000014,2018-12-31,2019-07-22,AssetsCurrent,"195,900.00",USD
4,1703157,"SECURETECH INNOVATIONS, INC.",0001703157-19-000017,2018-12-31,2019-10-21,AssetsCurrent,"195,900.00",USD
5,1703157,"SECURETECH INNOVATIONS, INC.",0001703157-20-000005,2018-12-31,2020-02-21,LiabilitiesCurrent,100.00,USD
6,1703157,"SECURETECH INNOVATIONS, INC.",0001703157-19-000003,2018-12-31,2019-02-19,AssetsCurrent,"195,900.00",USD
7,1703157,"SECURETECH INNOVATIONS, INC.",0001703157-20-000005,2018-12-31,2020-02-21,AssetsCurrent,"195,900.00",USD
8,1703157,"SECURETECH INNOVATIONS, INC.",0001703157-19-000014,2018-12-31,2019-07-22,LiabilitiesCurrent,100.00,USD
9,1703157,"SECURETECH INNOVATIONS, INC.",0001703157-19-000012,2018-12-31,2019-05-16,LiabilitiesCurrent,100.00,USD


CR extreme validation: Extreme values were traced to source-level NUM facts and are mathematically consistent with exceptionally low reported current liabilities relative to current assets. Similar observations occur across multiple filings of the same company, suggesting a genuine financial structure rather than a data or calculation error. Extreme CR observations are therefore retained; robust summaries will be used for aggregate analysis.

Cash Ratio

In [69]:
%%dsql 
SELECT DISTINCT 
    s.cik,
    s.name,
    accession_number,
    period_end,
    g.filed,
    n.tag,
    n.value,
    n.uom
FROM gold.financial_metrics g 
JOIN clean.num n 
ON g.accession_number = n.adsh
AND g.period_end=n.ddate
JOIN clean.sub s
ON g.accession_number = s.adsh
WHERE cash_ratio = (SELECT MAX(cash_ratio) FROM gold.financial_metrics) 
AND n.tag IN ('CashAndCashEquivalentsAtCarryingValue',
                'Cash',
                'ShortTermInvestments',
                'MarketableSecuritiesCurrent',
                'AvailableForSaleSecuritiesCurrent',
                'TradingSecuritiesCurrent',
                'LiabilitiesCurrent')

,cik,name,accession_number,period_end,filed,tag,value,uom
0,1703157,"SECURETECH INNOVATIONS, INC.",0001703157-19-000003,2018-12-31,2019-02-19,CashAndCashEquivalentsAtCarryingValue,"195,900.00",USD
1,1703157,"SECURETECH INNOVATIONS, INC.",0001703157-19-000014,2018-12-31,2019-07-22,LiabilitiesCurrent,100.00,USD
2,1703157,"SECURETECH INNOVATIONS, INC.",0001703157-19-000017,2018-12-31,2019-10-21,CashAndCashEquivalentsAtCarryingValue,"195,900.00",USD
3,1703157,"SECURETECH INNOVATIONS, INC.",0001703157-19-000014,2018-12-31,2019-07-22,CashAndCashEquivalentsAtCarryingValue,"195,900.00",USD
4,1703157,"SECURETECH INNOVATIONS, INC.",0001703157-20-000005,2018-12-31,2020-02-21,CashAndCashEquivalentsAtCarryingValue,"195,900.00",USD
5,1703157,"SECURETECH INNOVATIONS, INC.",0001703157-20-000005,2018-12-31,2020-02-21,LiabilitiesCurrent,100.00,USD
6,1703157,"SECURETECH INNOVATIONS, INC.",0001703157-19-000003,2018-12-31,2019-02-19,LiabilitiesCurrent,100.00,USD
7,1703157,"SECURETECH INNOVATIONS, INC.",0001703157-19-000017,2018-12-31,2019-10-21,LiabilitiesCurrent,100.00,USD


Quick Ratio

In [68]:
%%dsql 
SELECT DISTINCT 
    s.cik,
    s.name,
    accession_number,
    period_end,
    g.filed,
    n.tag,
    n.value,
    n.uom,
    n.qtrs,
    CASE WHEN quick_ratio = (SELECT MAX(quick_ratio) FROM gold.financial_metrics) THEN 1 ELSE 0 END AS is_max_quick_ratio
FROM gold.financial_metrics g 
JOIN clean.num n 
ON g.accession_number = n.adsh
AND g.period_end=n.ddate
JOIN clean.sub s
ON g.accession_number = s.adsh
WHERE (quick_ratio = (SELECT MAX(quick_ratio) FROM gold.financial_metrics) OR
       quick_ratio = (SELECT MIN(quick_ratio) FROM gold.financial_metrics))
AND n.tag IN ('AssetsCurrent', 'InventoryNet', 'PrepaidExpenseCurrent', 'LiabilitiesCurrent')
ORDER BY is_max_quick_ratio DESC, tag

,cik,name,accession_number,period_end,filed,tag,value,uom,qtrs,is_max_quick_ratio
0,1789029,"AEVA TECHNOLOGIES, INC.",0000950170-21-000296,2021-03-31,2021-06-02,AssetsCurrent,"526,694,000.00",USD,0,1
1,1789029,"AEVA TECHNOLOGIES, INC.",0000950170-21-000296,2021-03-31,2021-06-02,InventoryNet,"1,529,000.00",USD,0,1
2,1789029,"AEVA TECHNOLOGIES, INC.",0000950170-21-000296,2021-03-31,2021-06-02,LiabilitiesCurrent,"6,357,000.00",USD,0,1
3,1789029,"AEVA TECHNOLOGIES, INC.",0000950170-21-000296,2021-03-31,2021-06-02,PrepaidExpenseCurrent,"919,000.00",USD,0,1
4,1404804,OMNITEK ENGINEERING CORP,0001096906-24-001175,2024-03-31,2024-05-17,AssetsCurrent,"526,725.00",USD,0,0
5,1404804,OMNITEK ENGINEERING CORP,0001096906-24-001175,2024-03-31,2024-05-17,InventoryNet,"326,637.00",USD,0,0
6,1404804,OMNITEK ENGINEERING CORP,0001096906-24-001175,2024-03-31,2024-05-17,LiabilitiesCurrent,"1,696,222.00",USD,0,0
7,1404804,OMNITEK ENGINEERING CORP,0001096906-24-001175,2024-03-31,2024-05-17,PrepaidExpenseCurrent,"526,725.00",USD,0,0


Short Term Debt Ratio - A major change in it's max/min values after fixing the negative handling of the ratio.

In [34]:
%%dsql 
WITH a AS (
SELECT DISTINCT 
    s.cik,
    s.name,
    accession_number,
    period_end,
    n.tag,
    n.value,
    n.uom,
    short_term_debt_ratio,
    CASE WHEN 	short_term_debt_ratio = (SELECT MAX(short_term_debt_ratio) FROM gold.financial_metrics) THEN 1 ELSE 0 END AS is_max_short_term_debt_ratio,
    CASE WHEN short_term_debt_ratio = (SELECT MIN(short_term_debt_ratio) FROM gold.financial_metrics) THEN 1 ELSE 0 END AS is_min_short_term_debt_ratio
FROM gold.financial_metrics g 
JOIN clean.num n 
ON g.accession_number = n.adsh
AND g.period_end=n.ddate
JOIN clean.sub s
ON g.accession_number = s.adsh
WHERE (short_term_debt_ratio = (SELECT MAX(short_term_debt_ratio) FROM gold.financial_metrics) OR
       short_term_debt_ratio = (SELECT MIN(short_term_debt_ratio) FROM gold.financial_metrics))
AND qtrs = 0
AND uom = 'USD'
AND tag IN (
        'DebtCurrent',
        'ShortTermBorrowings',
        'LongTermDebtCurrent',
        'LongTermDebtAndCapitalLeaseObligationsCurrent',
        'FinanceLeaseLiabilityCurrent',
        'CapitalLeaseObligationsCurrent',
        'LongTermDebtAndCapitalLeaseObligations',
        'LongTermDebtNoncurrent',
        'FinanceLeaseLiabilityNoncurrent',
        'CapitalLeaseObligationsNoncurrent'
    )
)
SELECT *
FROM a
WHERE is_max_short_term_debt_ratio = 1 
ORDER BY is_max_short_term_debt_ratio DESC, name, tag

,cik,name,accession_number,period_end,tag,value,uom,short_term_debt_ratio,is_max_short_term_debt_ratio,is_min_short_term_debt_ratio
0,1818644,"AEYE, INC.",0001818644-23-000003,2022-12-31,ShortTermBorrowings,"8,594,000.00",USD,1.00,1,0
1,1530185,"AMERITEK VENTURES, INC.",0001376474-23-000405,2023-06-30,LongTermDebtCurrent,"80,109.00",USD,1.00,1,0
2,1530185,"AMERITEK VENTURES, INC.",0001376474-23-000492,2023-09-30,LongTermDebtCurrent,"89,097.00",USD,1.00,1,0
3,1530185,"AMERITEK VENTURES, INC.",0001376474-23-000302,2023-03-31,ShortTermBorrowings,"21,000.00",USD,1.00,1,0
4,1530185,"AMERITEK VENTURES, INC.",0001376474-24-000150,2022-12-31,ShortTermBorrowings,"21,000.00",USD,1.00,1,0
...,...,...,...,...,...,...,...,...,...,...
639,1096275,"WORKSPORT, LTD",0001493152-21-008647,2019-12-31,ShortTermBorrowings,"267,881.00",USD,1.00,1,0
640,1772720,XL FLEET CORP.,0001213900-21-019311,2020-12-31,DebtCurrent,"110,000.00",USD,1.00,1,0
641,1772720,XL FLEET CORP.,0001213900-21-019311,2019-12-31,DebtCurrent,"1,435,000.00",USD,1.00,1,0
642,1819493,"XOS, INC.",0001819493-22-000076,2021-12-31,CapitalLeaseObligationsCurrent,"482,000.00",USD,1.00,1,0


Calculation error on my end.

FCF/Debt Ratio

In [14]:
%%dsql 
WITH a AS (
SELECT DISTINCT 
    s.cik,
    s.name,
    accession_number,
    period_end,
    n.tag,
    n.value,
    n.uom,
    CASE WHEN fcf_to_debt_ratio	 = (SELECT MAX(fcf_to_debt_ratio) FROM gold.financial_metrics) THEN 1 ELSE 0 END AS is_max_fcf_to_debt_ratio,
    CASE WHEN fcf_to_debt_ratio	 = (SELECT MIN(fcf_to_debt_ratio) FROM gold.financial_metrics) THEN 1 ELSE 0 END AS is_min_fcf_to_debt_ratio
FROM gold.financial_metrics g 
JOIN clean.num n 
ON g.accession_number = n.adsh
AND g.period_end=n.ddate
JOIN clean.sub s
ON g.accession_number = s.adsh
WHERE (fcf_to_debt_ratio = (SELECT MAX(fcf_to_debt_ratio) FROM gold.financial_metrics) OR
       fcf_to_debt_ratio = (SELECT MIN(fcf_to_debt_ratio) FROM gold.financial_metrics))
AND (
        tag IN (
            'NetCashProvidedByUsedInOperatingActivities',
            'NetCashProvidedByUsedInOperatingActivitiesContinuingOperations',
            'PaymentsToAcquirePropertyPlantAndEquipment',
            'PaymentsToAcquireProductiveAssets',
            'CapitalExpenditures'
            )
            AND qtrs = 1
            AND uom = 'USD'
        )
        OR
        (
            tag IN (
                'DebtCurrent',
                'ShortTermBorrowings',
                'LongTermDebtCurrent',
                'NotesPayableCurrent',
                'FinanceLeaseLiabilityCurrent',
                'CapitalLeaseObligationsCurrent',
                'LongTermDebtAndCapitalLeaseObligationsCurrent',
                'LongTermDebtAndCapitalLeaseObligationsNoncurrent', -- Fixed: Included in filter array
                'LongTermDebtAndCapitalLeaseObligations',
                'LongTermDebtNoncurrent',
                'NotesPayableNoncurrent',
                'FinanceLeaseLiabilityNoncurrent',
                'CapitalLeaseObligationsNoncurrent'
            )
            AND qtrs = 0
            AND uom = 'USD'
        )
)
SELECT *
FROM a
WHERE is_min_fcf_to_debt_ratio = 1 OR 
      is_max_fcf_to_debt_ratio = 1
ORDER BY is_max_fcf_to_debt_ratio DESC, tag

,cik,name,accession_number,period_end,tag,value,uom,is_max_fcf_to_debt_ratio,is_min_fcf_to_debt_ratio
0,924822,MILLER INDUSTRIES INC /TN/,0000924822-21-000023,2021-03-31,FinanceLeaseLiabilityCurrent,22000.0,USD,1,0
1,924822,MILLER INDUSTRIES INC /TN/,0000924822-21-000023,2021-03-31,FinanceLeaseLiabilityNoncurrent,9000.0,USD,1,0
2,924822,MILLER INDUSTRIES INC /TN/,0000924822-21-000023,2021-03-31,NetCashProvidedByUsedInOperatingActivities,2847000.0,USD,1,0
3,924822,MILLER INDUSTRIES INC /TN/,0000924822-21-000023,2021-03-31,PaymentsToAcquirePropertyPlantAndEquipment,2489000.0,USD,1,0
4,1802749,"LIGHTNING EMOTORS, INC.",0001558370-22-008657,2022-03-31,FinanceLeaseLiabilityCurrent,61000.0,USD,0,1
5,1802749,"LIGHTNING EMOTORS, INC.",0001558370-22-008657,2022-03-31,FinanceLeaseLiabilityNoncurrent,299000.0,USD,0,1
6,1802749,"LIGHTNING EMOTORS, INC.",0001558370-22-008657,2022-03-31,NetCashProvidedByUsedInOperatingActivities,-16142000.0,USD,0,1
7,1802749,"LIGHTNING EMOTORS, INC.",0001558370-22-008657,2022-03-31,PaymentsToAcquirePropertyPlantAndEquipment,2024000.0,USD,0,1


Extreme-value validation — FCF-to-debt: The maximum and minimum observations were traced to their underlying SEC-reported cash-flow and debt components. Both extremes are reproducible from genuine source values: the maximum results from positive FCF combined with very low reported debt, while the minimum results from substantially negative FCF relative to debt. No evidence of a calculation or extraction anomaly was identified; therefore, the extreme observations are retained.

Return On Assets

In [16]:
%%dsql 
WITH a AS (
SELECT DISTINCT 
    s.cik,
    s.name,
    accession_number,
    period_end,
    n.tag,
    n.value,
    n.uom,
    CASE WHEN return_on_assets	 = (SELECT MAX(return_on_assets) FROM gold.financial_metrics) THEN 1 ELSE 0 END AS is_max_return_on_assets,
    CASE WHEN return_on_assets	 = (SELECT MIN(return_on_assets) FROM gold.financial_metrics) THEN 1 ELSE 0 END AS is_min_return_on_assets
FROM gold.financial_metrics g 
JOIN clean.num n 
ON g.accession_number = n.adsh
AND g.period_end=n.ddate
JOIN clean.sub s
ON g.accession_number = s.adsh
WHERE (return_on_assets = (SELECT MAX(return_on_assets) FROM gold.financial_metrics) OR
       return_on_assets = (SELECT MIN(return_on_assets) FROM gold.financial_metrics))
AND (
        (    tag IN (
                'NetIncomeLoss', 
                'ProfitLoss', 
                'NetIncomeLossAvailableToCommonStockholdersBasic', 
                'IncomeLossAttributableToParent',
                'NetIncomeLossAllocatedToLimitedPartners',
                'IncomeLossFromContinuingOperations'
            )
            AND qtrs = 1
            AND uom = 'USD'
        )
        OR
        (
            tag IN ('Assets', 'AssetsCurrent', 'AssetsNoncurrent')
            AND qtrs = 0
            AND uom = 'USD'
        )
    )
)
SELECT *
FROM a
WHERE is_min_return_on_assets = 1 OR 
      is_max_return_on_assets = 1
ORDER BY is_max_return_on_assets DESC, tag

,cik,name,accession_number,period_end,tag,value,uom,is_max_return_on_assets,is_min_return_on_assets
0,1404935,"THC THERAPEUTICS, INC.",0001477932-24-000075,2023-01-31,Assets,23030.0,USD,1,0
1,1404935,"THC THERAPEUTICS, INC.",0001477932-24-000075,2023-01-31,AssetsCurrent,10329.0,USD,1,0
2,1404935,"THC THERAPEUTICS, INC.",0001477932-24-000075,2023-01-31,NetIncomeLoss,1685852.0,USD,1,0
3,1404935,"THC THERAPEUTICS, INC.",0001477932-24-000075,2023-01-31,ProfitLoss,1685852.0,USD,1,0
4,1404935,"THC THERAPEUTICS, INC.",0001477932-19-003523,2019-04-30,Assets,101007.0,USD,0,1
5,1404935,"THC THERAPEUTICS, INC.",0001477932-19-003523,2019-04-30,AssetsCurrent,32344.0,USD,0,1
6,1404935,"THC THERAPEUTICS, INC.",0001477932-19-003523,2019-04-30,NetIncomeLoss,-22131089.0,USD,0,1


ROA extreme validation

Max and min values traced to source NUM observations.
Both extremes originate from the same company but different reporting periods.
Extreme values are explained by unusually large net income/loss relative to asset base.
No evidence of malformed source values or calculation error.
Decision: retain extremes.
Analytical implication: ROA is highly susceptible to extreme values; median/percentiles should accompany mean in aggregate summaries.

Cash Earnings Conversion

In [12]:
%%dsql 
WITH a AS (
SELECT DISTINCT 
    s.cik,
    s.name,
    accession_number,
    period_end,
    n.tag,
    n.value,
    n.uom,
    cash_earnings_conversion,
    CASE WHEN 	cash_earnings_conversion = (SELECT MAX(cash_earnings_conversion) FROM gold.financial_metrics) THEN 1 ELSE 0 END AS is_max_cash_earnings_conversion,
    CASE WHEN cash_earnings_conversion = (SELECT MIN(cash_earnings_conversion) FROM gold.financial_metrics) THEN 1 ELSE 0 END AS is_min_cash_earnings_conversion
FROM gold.financial_metrics g 
JOIN clean.num n 
ON g.accession_number = n.adsh
AND g.period_end=n.ddate
JOIN clean.sub s
ON g.accession_number = s.adsh
WHERE (cash_earnings_conversion = (SELECT MAX(cash_earnings_conversion) FROM gold.financial_metrics) OR
       cash_earnings_conversion = (SELECT MIN(cash_earnings_conversion) FROM gold.financial_metrics))
AND tag IN (
    'NetCashProvidedByUsedInOperatingActivities', 
    'NetCashProvidedByUsedInOperatingActivitiesContinuingOperations',
    'NetIncomeLoss', 
    'ProfitLoss', 
    'NetIncomeLossAvailableToCommonStockholdersBasic', 
    'IncomeLossAttributableToParent',
    'NetIncomeLossAllocatedToLimitedPartners',
    'IncomeLossFromContinuingOperations'
)
)
SELECT *
FROM a
WHERE is_min_cash_earnings_conversion = 1 OR 
      is_max_cash_earnings_conversion = 1
ORDER BY is_max_cash_earnings_conversion DESC, tag

,cik,name,accession_number,period_end,tag,value,uom,cash_earnings_conversion,is_max_cash_earnings_conversion,is_min_cash_earnings_conversion
0,1024095,"CARNEGIE DEVELOPMENT, INC",0001477932-20-002748,2019-03-31,NetCashProvidedByUsedInOperatingActivities,-119916.0,USD,722.39,1,0
1,1024095,"CARNEGIE DEVELOPMENT, INC",0001477932-19-005096,2019-03-31,NetCashProvidedByUsedInOperatingActivities,-119916.0,USD,722.39,1,0
2,1024095,"CARNEGIE DEVELOPMENT, INC",0001477932-20-002748,2019-03-31,NetIncomeLoss,-166.0,USD,722.39,1,0
3,1024095,"CARNEGIE DEVELOPMENT, INC",0001477932-19-005096,2019-03-31,NetIncomeLoss,-166.0,USD,722.39,1,0
4,775158,OSHKOSH CORP,0000950170-23-015169,2022-03-31,NetCashProvidedByUsedInOperatingActivities,328900000.0,USD,-1644.50,0,1
5,775158,OSHKOSH CORP,0000950170-23-015169,2022-03-31,ProfitLoss,-200000.0,USD,-1644.50,0,1


Cash Earnings Conversion — extremes validated. The maximum (722.39) and minimum (−1,644.50) were traced to their underlying OCF and net-income facts in NUM. Recalculation from the source values reproduces the Gold-layer ratios, confirming that the extreme values arise from genuine financial values rather than an apparent calculation or extraction error.

OCF/Debt Ratio

In [15]:
%%dsql 
WITH a AS (
SELECT DISTINCT 
    s.cik,
    s.name,
    accession_number,
    period_end,
    n.tag,
    n.value,
    n.uom,
    ocf_debt_ratio,
    CASE WHEN ocf_debt_ratio = (SELECT MAX(ocf_debt_ratio) FROM gold.financial_metrics) THEN 1 ELSE 0 END AS is_max_ocf_debt_ratio,
    CASE WHEN ocf_debt_ratio = (SELECT MIN(ocf_debt_ratio) FROM gold.financial_metrics) THEN 1 ELSE 0 END AS is_min_ocf_debt_ratio
FROM gold.financial_metrics g 
JOIN clean.num n 
ON g.accession_number = n.adsh
AND g.period_end=n.ddate
JOIN clean.sub s
ON g.accession_number = s.adsh
WHERE (ocf_debt_ratio = (SELECT MAX(ocf_debt_ratio) FROM gold.financial_metrics) OR
       ocf_debt_ratio = (SELECT MIN(ocf_debt_ratio) FROM gold.financial_metrics))
AND (
        ( tag IN (
            'NetCashProvidedByUsedInOperatingActivities', 
            'NetCashProvidedByUsedInOperatingActivitiesContinuingOperations')
            AND qtrs = 1
        )
        OR
        (
            tag IN (
                'ShortTermBorrowings',
                'LongTermDebtCurrent',
                'LongTermDebtNoncurrent'
            )
            AND qtrs = 0
        )
    )
)
SELECT *
FROM a
WHERE is_min_ocf_debt_ratio = 1 OR 
      is_max_ocf_debt_ratio = 1
ORDER BY is_max_ocf_debt_ratio DESC, tag

,cik,name,accession_number,period_end,tag,value,uom,ocf_debt_ratio,is_max_ocf_debt_ratio,is_min_ocf_debt_ratio
0,1670541,ADIENT PLC,0001670541-20-000004,2019-12-31,NetCashProvidedByUsedInOperatingActivities,239000000.0,USD,39.83,1,0
1,1670541,ADIENT PLC,0001670541-20-000004,2019-12-31,ShortTermBorrowings,6000000.0,USD,39.83,1,0
2,1498148,ARTIFICIAL INTELLIGENCE TECHNOLOGY SOLUTIONS INC.,0001161697-22-000361,2022-05-31,NetCashProvidedByUsedInOperatingActivities,-3621572.0,USD,-2272.00,0,1
3,1498148,ARTIFICIAL INTELLIGENCE TECHNOLOGY SOLUTIONS INC.,0001161697-22-000361,2022-05-31,ShortTermBorrowings,1594.0,USD,-2272.00,0,1


OCF/debt extremes are genuine and mathematically consistent with the underlying source facts.

Interest Coverage Ratio - Before

In [16]:
%%dsql 
WITH a AS (
SELECT DISTINCT 
    s.cik,
    s.name,
    accession_number,
    period_end,
    n.tag,
    n.value,
    n.uom,
    interest_coverage_ratio,
    CASE WHEN interest_coverage_ratio = (SELECT MAX(interest_coverage_ratio) FROM gold.financial_metrics) THEN 1 ELSE 0 END AS is_max_interest_coverage_ratio,
    CASE WHEN interest_coverage_ratio = (SELECT MIN(interest_coverage_ratio) FROM gold.financial_metrics) THEN 1 ELSE 0 END AS is_min_interest_coverage_ratio
FROM gold.financial_metrics g 
JOIN clean.num n 
ON g.accession_number = n.adsh
AND g.period_end=n.ddate
JOIN clean.sub s
ON g.accession_number = s.adsh
WHERE (interest_coverage_ratio = (SELECT MAX(interest_coverage_ratio) FROM gold.financial_metrics) OR
       interest_coverage_ratio = (SELECT MIN(interest_coverage_ratio) FROM gold.financial_metrics))
AND n.tag IN (
        'OperatingIncomeLoss', 
        'IncomeLossFromContinuingOperationsBeforeIncomeTaxesMinorityInterestAndIncomeTaxes',
        'InterestExpense', 
        'InterestExpenseDebt', 
        'InterestAndDebtExpense', 
        'InterestExpenseNet'
    )
    AND (
        (s.fp IN ('Q1', 'Q2', 'Q3') AND n.qtrs = 1)
        OR 
        (s.fp = 'FY' AND n.qtrs = 4)
    )
)
SELECT *
FROM a
WHERE is_min_interest_coverage_ratio = 1 OR 
      is_max_interest_coverage_ratio = 1
ORDER BY is_max_interest_coverage_ratio DESC, tag

,cik,name,accession_number,period_end,tag,value,uom,interest_coverage_ratio,is_max_interest_coverage_ratio,is_min_interest_coverage_ratio
0,1707919,CENNTRO INC.,0001140361-24-036925,2023-06-30,InterestExpense,-1262.0,USD,10319.96,1,0
1,1707919,CENNTRO INC.,0001140361-24-036925,2023-06-30,OperatingIncomeLoss,-13023787.0,USD,10319.96,1,0
2,1811210,"LUCID GROUP, INC.",0001628280-22-012681,2021-03-31,InterestExpense,5000.0,USD,-59758.60,0,1
3,1811210,"LUCID GROUP, INC.",0001628280-22-012681,2021-03-31,OperatingIncomeLoss,-298793000.0,USD,-59758.60,0,1


Interest Coverage Ratio - After

In [42]:
%%dsql 
WITH a AS (
SELECT DISTINCT 
    s.cik,
    s.name,
    accession_number,
    period_end,
    n.tag,
    n.value,
    n.uom,
    interest_coverage_ratio,
    CASE WHEN interest_coverage_ratio = (SELECT MAX(interest_coverage_ratio) FROM gold.financial_metrics) THEN 1 ELSE 0 END AS is_max_interest_coverage_ratio,
    CASE WHEN interest_coverage_ratio = (SELECT MIN(interest_coverage_ratio) FROM gold.financial_metrics) THEN 1 ELSE 0 END AS is_min_interest_coverage_ratio
FROM gold.financial_metrics g 
JOIN clean.num n 
ON g.accession_number = n.adsh
AND g.period_end=n.ddate
JOIN clean.sub s
ON g.accession_number = s.adsh
WHERE (interest_coverage_ratio = (SELECT MAX(interest_coverage_ratio) FROM gold.financial_metrics) OR
       interest_coverage_ratio = (SELECT MIN(interest_coverage_ratio) FROM gold.financial_metrics))
AND n.tag IN (
        'OperatingIncomeLoss', 
        'IncomeLossFromContinuingOperationsBeforeIncomeTaxesMinorityInterestAndIncomeTaxes',
        'InterestExpense', 
        'InterestExpenseDebt', 
        'InterestAndDebtExpense', 
        'InterestExpenseNet'
    )
    AND (
        (s.fp IN ('Q1', 'Q2', 'Q3') AND n.qtrs = 1)
        OR 
        (s.fp = 'FY' AND n.qtrs = 4)
    )
)
SELECT *
FROM a
WHERE is_min_interest_coverage_ratio = 1 OR 
      is_max_interest_coverage_ratio = 1
ORDER BY is_max_interest_coverage_ratio DESC, tag

,cik,name,accession_number,period_end,tag,value,uom,interest_coverage_ratio,is_max_interest_coverage_ratio,is_min_interest_coverage_ratio
0,743238,"SHYFT GROUP, INC.",0001437749-21-025301,2020-09-30,InterestExpense,"11,000.00",USD,"2,400.09",1,0
1,743238,"SHYFT GROUP, INC.",0001437749-20-022774,2020-09-30,InterestExpenseDebt,"11,000.00",USD,"2,400.09",1,0
2,743238,"SHYFT GROUP, INC.",0001437749-21-025301,2020-09-30,OperatingIncomeLoss,"26,401,000.00",USD,"2,400.09",1,0
3,743238,"SHYFT GROUP, INC.",0001437749-20-022774,2020-09-30,OperatingIncomeLoss,"26,401,000.00",USD,"2,400.09",1,0
4,1811210,"LUCID GROUP, INC.",0001628280-22-012681,2021-03-31,InterestExpense,"5,000.00",USD,"-59,758.60",0,1
5,1811210,"LUCID GROUP, INC.",0001628280-22-012681,2021-03-31,OperatingIncomeLoss,"-298,793,000.00",USD,"-59,758.60",0,1


Validation finding — treatment of negative financial values

During extreme-value validation of the financial ratios, the Interest Coverage Ratio (ICR) produced unusually large positive values. Investigation of the underlying filing data showed that both the numerator and denominator could be negative. Because direct division of two negative values produces a positive ratio, this could result in a mathematically valid but potentially misleading interpretation of the underlying financial condition.

This prompted a broader review of the treatment of negative numerator/denominator values across the ratio calculations. The ratio logic was subsequently revised to explicitly handle the sign combinations rather than implicitly allowing negative values to cancel through ordinary division.

The revised calculations were then compared against the original EDA summaries. Most metrics remained unchanged or showed negligible differences, while metrics particularly sensitive to negative values showed changes in their distributions and extreme values. For example, the maximum ICR changed from 10,319.96 to 2,400.09 after the revised calculation logic.

This validation step therefore served not only as an outlier investigation, but also as a check on the financial interpretability of the calculated metrics.

After revising the ratio logic to explicitly handle negative numerator and denominator values, the extreme ICR observations were re-evaluated against their underlying filing values. The revised maximum ICR of 2,400.09 corresponds to SHYFT Group, with positive operating income of $26.4M and interest expense of $11K. The minimum ICR of −59,758.60 corresponds to Lucid Group, where operating loss of approximately $298.8M is paired with $5K of interest expense. Both observations are consistent with the underlying reported values; no apparent source-data anomaly was identified.

Summary Before

In [71]:
%%dsql
SELECT 
    column_name AS metric,
    ROUND(CAST(min AS NUMERIC),2) AS min,
    ROUND(CAST(max AS NUMERIC),2) AS max,
    ROUND(CAST(avg AS NUMERIC),2) AS mean,
    ROUND(CAST(std AS NUMERIC),2) AS std_dev,
    ROUND(CAST(q25 AS NUMERIC),2) AS q25,
    ROUND(CAST(q50 AS NUMERIC),2) AS median,
    ROUND(CAST(q75 AS FLOAT),2) AS q75
FROM (SUMMARIZE TABLE gold.financial_metrics)t
WHERE column_name NOT IN ('accession_number','period_end','filed')
ORDER BY min DESC;
## Summary Before

,metric,min,max,mean,std_dev,q25,median,q75
0,current_ratio,0.00,"1,959.00",7.83,86.81,1.27,1.72,2.49
1,cash_ratio,0.00,"1,959.00",5.75,81.55,0.08,0.27,0.61
2,debt_to_assets_ratio,0.00,7.65,0.14,0.25,0.00,0.01,0.23
3,short_term_debt_ratio,0.00,1.00,0.23,0.34,0.01,0.04,0.27
4,quick_ratio,-0.19,82.47,2.81,5.63,0.85,1.27,1.83
5,fcf_to_debt_ratio,-50.46,11.55,-0.74,4.21,-0.13,-0.03,0.02
6,return_on_assets,-219.10,73.20,-0.36,6.16,-0.03,0.01,0.02
7,cash_earnings_conversion,"-1,644.50",722.39,0.04,28.79,0.00,0.00,0.00
8,ocf_debt_ratio,"-2,272.00",39.83,-24.69,213.40,-0.20,-0.01,0.06
9,interest_coverage_ratio,"-59,758.60","10,319.96",-70.63,"1,521.16",-1.59,1.61,8.46


Summary After

In [ ]:
%%dsql
SELECT 
    column_name AS metric,
    ROUND(CAST(min AS NUMERIC),2) AS min,
    ROUND(CAST(max AS NUMERIC),2) AS max,
    ROUND(CAST(avg AS NUMERIC),2) AS mean,
    ROUND(CAST(std AS NUMERIC),2) AS std_dev,
    ROUND(CAST(q25 AS NUMERIC),2) AS q25,
    ROUND(CAST(q50 AS NUMERIC),2) AS median,
    ROUND(CAST(q75 AS FLOAT),2) AS q75
FROM (SUMMARIZE TABLE gold.financial_metrics)t
WHERE column_name NOT IN ('accession_number','period_end','filed')
ORDER BY min DESC;
## Summary After

,metric,min,max,mean,std_dev,q25,median,q75
0,current_ratio,0.00,"1,959.00",7.83,86.81,1.27,1.72,2.49
1,cash_ratio,0.00,"1,959.00",5.75,81.55,0.08,0.27,0.61
2,debt_to_assets_ratio,0.00,7.65,0.14,0.25,0.00,0.01,0.23
3,short_term_debt_ratio,0.00,1.00,0.23,0.34,0.01,0.04,0.27
4,quick_ratio,-0.19,82.47,2.81,5.63,0.85,1.27,1.83
5,fcf_to_debt_ratio,-50.46,11.55,-0.74,4.21,-0.13,-0.03,0.02
6,return_on_assets,-219.10,73.20,-0.36,6.16,-0.03,0.01,0.02
7,cash_earnings_conversion,"-1,644.50",649.49,-1.00,28.77,0.00,0.00,0.00
8,ocf_debt_ratio,"-2,272.00",39.83,-24.69,213.40,-0.20,-0.01,0.06
9,interest_coverage_ratio,"-59,758.60","2,400.09",-77.10,"1,520.85",-1.70,1.55,8.40
